In [1]:
"""
Refactor du feature extractor vidéo avec:
- Couverture temporelle >16 frames pour MViT via chunking (fenêtres de 16) et agrégation.
- Stratégies d'échantillonnage: aléatoire ou uniformément réparties.
- Extraction centrale via ResNet (frame médiane).
- Embedding optical flow (TV-L1 OpenCV) + encodeur CNN (ResNet18).
- Hooks pour distillation (si disponible) ou fine-tuning léger (dé-gel partiel + MLP).


Dépendances suggérées:
- torch, torchvision
- pytorchvideo (optionnel mais recommandé pour MViT)
- opencv-contrib-python (pour TV-L1): `cv2.optflow`


Note: le code est structuré pour être plug-and-play et modulaire. Les endroits
qui dépendent de votre infra (chemins MSR-VTT, dataloader exact, etc.) sont marqués TODO.
"""
from dataclasses import dataclass
from typing import List, Literal, Optional, Tuple, Dict
import math
import random

import os
import numpy as np


import torch
import torch.nn as nn
import torch.nn.functional as F


try:
    import torchvision
    from torchvision.transforms import functional as TF
except Exception as e:
    torchvision = None
    TF = None


try:
    # PytorchVideo pour MViT
    from pytorchvideo.models.hub import mvit_base_16x4
except Exception:
    mvit_base_16x4 = None


try:
    import cv2
    HAS_CV2 = True
except Exception:
    HAS_CV2 = False

In [2]:
# ---------------------- Sampling utils ----------------------

@dataclass
class SamplingConfig:
    num_frames: int = 64
    strategy: Literal["uniform", "random"] = "uniform"
    seed: Optional[int] = None


def sample_indices(n_total: int, cfg: SamplingConfig) -> List[int]:
    """Sélectionne des indices de frames en [0, n_total-1] selon la stratégie.
    - uniform: répartit équitablement sur la durée
    - random: échantillonne sans remise
    Retourne une liste triée croissante.
    """
    if n_total <= 0:
        return []

    k = min(cfg.num_frames, n_total)
    if cfg.strategy == "uniform":
        if k == 1:
            return [n_total // 2]
        step = (n_total - 1) / (k - 1)
        idx = [int(round(i * step)) for i in range(k)]
        return sorted(set(idx))
    else:
        rng = random.Random(cfg.seed)
        idx = sorted(rng.sample(range(n_total), k))
        return idx

In [3]:
"""

# ---------------------- Frame decoding (placeholder) ----------------------

class SimpleVideoReader:
    #""#"Très léger wrapper autour d'opencv VideoCapture.
    #Remplacez par votre décodage (decord, pyav) si nécessaire.
    #""#"
    def __init__(self, path: str):
        if not HAS_CV2:
            raise ImportError("OpenCV n'est pas disponible. Installez opencv-python-headless ou opencv-contrib-python.")
        self.cap = cv2.VideoCapture(path)
        if not self.cap.isOpened():
            raise IOError(f"Impossible d'ouvrir la vidéo: {path}")
        self.length = int(self.cap.get(cv2.CAP_PROP_FRAME_COUNT))

    def __len__(self):
        return self.length

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc, tb):
        try:
            self.release()
        except Exception:
            pass

    def release(self):
        if getattr(self, "cap", None) is not None:
            try:
                self.cap.release()
            except Exception:
                pass

    def read_frames(self, indices: List[int]) -> List["np.ndarray"]:
        frames = []
        for i in indices:
            self.cap.set(cv2.CAP_PROP_POS_FRAMES, i)
            ok, frame = self.cap.read()
            if not ok:
                continue
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)
        return frames

"""

'\n\n# ---------------------- Frame decoding (placeholder) ----------------------\n\nclass SimpleVideoReader:\n    #""#"Très léger wrapper autour d\'opencv VideoCapture.\n    #Remplacez par votre décodage (decord, pyav) si nécessaire.\n    #""#"\n    def __init__(self, path: str):\n        if not HAS_CV2:\n            raise ImportError("OpenCV n\'est pas disponible. Installez opencv-python-headless ou opencv-contrib-python.")\n        self.cap = cv2.VideoCapture(path)\n        if not self.cap.isOpened():\n            raise IOError(f"Impossible d\'ouvrir la vidéo: {path}")\n        self.length = int(self.cap.get(cv2.CAP_PROP_FRAME_COUNT))\n\n    def __len__(self):\n        return self.length\n\n    def __enter__(self):\n        return self\n\n    def __exit__(self, exc_type, exc, tb):\n        try:\n            self.release()\n        except Exception:\n            pass\n\n    def release(self):\n        if getattr(self, "cap", None) is not None:\n            try:\n                self.

In [4]:
# ---------------------- Frame decoding (PyAV + GPU / CUDA) ----------------------
import av
import torch
import numpy as np
from typing import List

class SimpleVideoReader:
    """
    High-speed video reader using PyAV.
    - Essaie le décodage GPU (FFmpeg + NVDEC/CUDA), fallback CPU sinon.
    - API compatible avec l'ancienne version Decord.
    - read_frames(indices) -> List[np.ndarray] (H, W, C) par défaut
      ou List[torch.Tensor] sur CUDA si output_as_tensor=True.
    """

    def __init__(self, path: str, use_gpu: bool = True, output_as_tensor: bool = False):
        self.path = path
        self.use_gpu = use_gpu
        self.output_as_tensor = output_as_tensor
        self.container = None
        self.stream = None

        # Ouverture du container avec tentative d'accélération matérielle
        try:
            if use_gpu:
                # Requiert un FFmpeg compilé avec CUDA/NVDEC.
                self.container = av.open(
                    path,
                    options={"hwaccel": "cuda", "hwaccel_device": "0"}
                )
                #print("✅ PyAV: décodage GPU demandé (CUDA/NVDEC).")
            else:
                self.container = av.open(path)
        except av.AVError:
            #print("⚠️ PyAV: ouverture avec GPU impossible, passage CPU.")
            self.container = av.open(path)
            self.use_gpu = False

        # Stream vidéo principal
        self.stream = self.container.streams.video[0]
        self.fps = float(self.stream.average_rate) if self.stream.average_rate else 0.0
        self.time_base = float(self.stream.time_base) if self.stream.time_base else 1.0

        # Longueur (si dispo) ; sinon on laissera 0 (non bloquant pour l’API existante)
        self.length = int(self.stream.frames) if self.stream.frames else 0

    def __len__(self):
        return self.length

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc, tb):
        self.release()

    def release(self):
        try:
            if self.container is not None:
                self.container.close()
        except Exception:
            pass

    # --- utilitaire : seek vers une frame précise ---
    def _seek_to_frame(self, frame_idx: int):
        """
        Convertit un index de frame en timestamp PTS selon le time_base du stream,
        puis positionne le lecteur au plus proche frame précédent.
        """
        if self.fps <= 0 or self.time_base <= 0:
            # Seek au début comme fallback
            self.container.seek(0, stream=self.stream)
            return
        # pts = t / time_base, t = frame_idx / fps
        pts = int((frame_idx / self.fps) / self.time_base)
        self.container.seek(pts, stream=self.stream, any_frame=False, backward=True)

    def read_frames(self, indices: List[int]) -> List[np.ndarray]:
        """
        Lit une liste d'indices de frames (0-based).
        Renvoie une liste de frames au format HWC (RGB).
        Si output_as_tensor=True, chaque frame est un torch.Tensor sur CUDA.
        """
        if not indices:
            return []

        # On trie et on déduplique pour optimiser les seeks
        sorted_unique = sorted(set(int(i) for i in indices))
        wanted = set(sorted_unique)
        last_needed = sorted_unique[-1]

        out = []
        collected = {}

        # On fait un seek au premier index demandé
        self._seek_to_frame(sorted_unique[0])

        current_idx = -1
        for frame in self.container.decode(video=0):  # <- VideoFrame direct, pas de boucle interne
            current_idx += 1
            # Arrêt anticipé une fois le plus grand index dépassé
            if current_idx > last_needed and len(collected) == len(wanted):
                break

            if current_idx in wanted and current_idx not in collected:
                # Conversion en RGB (HWC)
                arr = frame.to_ndarray(format="rgb24")  # ndarray CPU (H, W, 3)

                if self.output_as_tensor:
                    # Copie sur GPU: tensor HWC (on garde la compatibilité avec l'API précédente)
                    t = torch.from_numpy(arr).to("cuda", non_blocking=True)
                    collected[current_idx] = t
                else:
                    collected[current_idx] = arr

        # Restituer dans l'ordre demandé (même si doublons)
        for i in indices:
            if i in collected:
                out.append(collected[i])
            else:
                # si non trouvé (seek imprécis ou fin de vidéo), on ignore ou on peut lever
                # Ici on préfère ignorer silencieusement pour rester tolérant.
                pass

        return out


In [5]:
# ---------------------- Preprocess ----------------------

@dataclass
class PreprocessConfig:
    size: int = 224
    center_crop: bool = True
    mean: Tuple[float, float, float] = (0.485, 0.456, 0.406)
    std: Tuple[float, float, float] = (0.229, 0.224, 0.225)


def preprocess_frame(img, cfg: PreprocessConfig) -> torch.Tensor:
    """img: HxWx3 (RGB, uint8). Retourne Tensor 3xHxW normalisé."""
    # Convert to tensor
    x = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
    # Resize shortest side -> cfg.size
    _, h, w = x.shape
    scale = cfg.size / min(h, w)
    new_h, new_w = int(round(h * scale)), int(round(w * scale))
    x = F.interpolate(x.unsqueeze(0), size=(new_h, new_w), mode="bilinear", align_corners=False).squeeze(0)
    # Center crop or simple crop
    if cfg.center_crop:
        top = (new_h - cfg.size) // 2
        left = (new_w - cfg.size) // 2
        x = x[:, top: top + cfg.size, left: left + cfg.size]
    else:
        # simple top-left crop
        x = x[:, :cfg.size, :cfg.size]
    # Normalize
    mean = torch.tensor(cfg.mean).view(3, 1, 1)
    std = torch.tensor(cfg.std).view(3, 1, 1)
    x = (x - mean) / std
    return x

In [6]:
# ---------------------- Backbones ----------------------

class MViTBackbone(nn.Module):
    """Wrap MViT (pré-entraîné) et accepte des clips de 16 frames.
    Pour >16 frames: on découpe en chunks de 16 et on agrège (mean/attn-pool).
    """
    def __init__(self, feature_dim: int = 768, pretrained: bool = True, agg: Literal["mean", "max"] = "mean"):
        super().__init__()
        if mvit_base_16x4 is None:
            raise ImportError("pytorchvideo non disponible. Installez pytorchvideo pour MViT.")
        self.model = mvit_base_16x4(pretrained=pretrained)
        # Supprimer la tête de classification pour obtenir des features
        if hasattr(self.model, "blocks"):  # pytorchvideo mvit
            if hasattr(self.model, "head") and hasattr(self.model.head, "proj"):
                self.model.head.proj = nn.Identity()
        self.agg = agg
        self.feature_dim = feature_dim

    def forward_clip16(self, clip: torch.Tensor) -> torch.Tensor:
        """clip: (B, C=3, T=16, H, W) -> (B, D)"""
        out = self.model(clip)
        if isinstance(out, (list, tuple)):
            out = out[0]
        return out

    def forward_many(self, clip: torch.Tensor) -> torch.Tensor:
        """clip: (B, C, T, H, W) avec T>=1. Découpe en segments de 16.
        Retour: (B, D)
        """
        B, C, T, H, W = clip.shape
        if T <= 16:
            if T < 16:
                # pad en répétant la dernière frame
                pad = 16 - T
                last = clip[:, :, -1:, :, :].repeat(1, 1, pad, 1, 1)
                clip = torch.cat([clip, last], dim=2)
            return self.forward_clip16(clip)
        # découpage
        chunks = []
        for start in range(0, T, 16):
            seg = clip[:, :, start: start + 16, :, :]
            if seg.shape[2] < 16:
                pad = 16 - seg.shape[2]
                last = seg[:, :, -1:, :, :].repeat(1, 1, pad, 1, 1)
                seg = torch.cat([seg, last], dim=2)
            feat = self.forward_clip16(seg)
            chunks.append(feat)
        feats = torch.stack(chunks, dim=1)  # (B, S, D)
        if self.agg == "mean":
            return feats.mean(dim=1)
        else:
            return feats.max(dim=1).values


class ResNetBackbone(nn.Module):
    def __init__(self, arch: str = "resnet50", pretrained: bool = True):
        super().__init__()
        if torchvision is None:
            raise ImportError("torchvision non disponible.")
        weights = None
        if pretrained:
            # Try new API first
            try:
                from torchvision.models import get_model_weights
                weights = get_model_weights(arch).DEFAULT
            except Exception:
                # Fallback to known enums for common resnets
                try:
                    if arch == "resnet50":
                        weights = torchvision.models.ResNet50_Weights.DEFAULT
                    elif arch == "resnet18":
                        weights = torchvision.models.ResNet18_Weights.DEFAULT
                    else:
                        weights = None
                except Exception:
                    weights = None
        # Create base model with best-effort weights across torchvision versions
        try:
            base = getattr(torchvision.models, arch)(weights=weights)
        except TypeError:
            base = getattr(torchvision.models, arch)(pretrained=bool(weights))
        # Keep encoder up to global pool
        self.encoder = nn.Sequential(*list(base.children())[:-1])  # global pool
        # Infer output dim from the final FC input features
        self.out_dim = getattr(base.fc, "in_features", 2048)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, 3, H, W) -> (B, D)
        feat = self.encoder(x)
        return feat.flatten(1)

In [7]:
# ===== NVIDIA Optical Flow (GPU) with TV-L1 (CPU) fallback =====
try:
    import cv2
    HAS_CV2 = True
except Exception:
    HAS_CV2 = False
    cv2 = None  # type: ignore

class OpticalFlowComputer:
    """
    Wraps NVIDIA Optical Flow (GPU) if available; otherwise falls back to TV-L1 (CPU).
    - Requires RTX-class GPU for NVOF.
    - Reuses the instantiated backend across frames.
    """
    def __init__(self):
        self.use_nvof = False
        self.nvof = None
        self.tvl1 = None

        if not HAS_CV2:
            raise RuntimeError("OpenCV is required for optical flow. Please install opencv-contrib-python-headless.")

        # Try GPU NVIDIA Optical Flow first
        try:
            has_cuda = hasattr(cv2, "cuda") and cv2.cuda.getCudaEnabledDeviceCount() > 0
            has_nvof = hasattr(cv2, "cuda_NvidiaOpticalFlow_1_0")
            if has_cuda and has_nvof:
                self.nvof = cv2.cuda_NvidiaOpticalFlow_1_0.create()
                self.use_nvof = True
        except Exception:
            self.use_nvof = False
            self.nvof = None

        # Fallback to CPU TV-L1 if NVOF not available
        if not self.use_nvof:
            if not hasattr(cv2, "optflow") or not hasattr(cv2.optflow, "DualTVL1OpticalFlow_create"):
                raise RuntimeError(
                    "Neither NVIDIA Optical Flow (GPU) nor TV-L1 (CPU) is available.\n"
                    "Install an OpenCV build with CUDA/NVOF support for GPU or opencv-contrib for TV-L1."
                )
            self.tvl1 = OpticalFlowComputer()

    def compute(self, prev_gray, gray):
        """
        prev_gray, gray: uint8 HxW single-channel numpy arrays
        returns: flow as HxWx2 float32 numpy array
        """
        if self.use_nvof:
            gpu_prev = cv2.cuda_GpuMat()
            gpu_gray = cv2.cuda_GpuMat()
            gpu_prev.upload(prev_gray)
            gpu_gray.upload(gray)
            flow_gpu = self.nvof.calc(gpu_prev, gpu_gray, None)
            flow = flow_gpu.download()
            return flow
        else:
            return self.tvl1.calc(prev_gray, gray, None)
# ===== End of optical flow wrapper =====

# ---------------------- Optical Flow Embedding ----------------------

class OpticalFlowEncoder(nn.Module):
    """Calcule le flow TV-L1 et l'encode via un ResNet18.
    Entrée: liste de frames RGB uint8 (HxWx3). Sortie: (B, D_flow)
    """
    def __init__(self, pretrained: bool = True):
        super().__init__()
        if torchvision is None:
            raise ImportError("torchvision non disponible.")
        base = torchvision.models.resnet18(weights=(torchvision.models.ResNet18_Weights.DEFAULT if pretrained else None))
        # adapter 3 canaux pour [flow_x, flow_y, magnitude]
        self.stem = base.conv1
        if self.stem.in_channels != 3:
            self.stem = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        base.conv1 = self.stem
        self.encoder = nn.Sequential(*list(base.children())[:-1])
        self.out_dim = 512

    @staticmethod
    def _compute_tv_l1(prev, nxt) -> Tuple["np.ndarray", "np.ndarray"]:
        assert HAS_CV2, "OpenCV requis pour le flow."
        gray1 = cv2.cvtColor(prev, cv2.COLOR_RGB2GRAY)
        gray2 = cv2.cvtColor(nxt, cv2.COLOR_RGB2GRAY)
        # Try multiple constructors for TV-L1, else fallback to Farneback
        try:
            if hasattr(cv2, "optflow") and hasattr(cv2.optflow, "DualTVL1OpticalFlow_create"):
                tvl1 = OpticalFlowComputer()
            elif hasattr(cv2, "DualTVL1OpticalFlow_create"):
                tvl1 = cv2.DualTVL1OpticalFlow_create()
            elif hasattr(cv2, "optflow") and hasattr(cv2.optflow, "createOptFlow_DualTVL1"):
                tvl1 = cv2.optflow.createOptFlow_DualTVL1()
            else:
                raise AttributeError("TV-L1 non disponible")
            flow = tvl1.calc(gray1, gray2, None)  # HxWx2 (fx, fy)
        except Exception:
            # Fallback: Farneback dense
            flow = cv2.calcOpticalFlowFarneback(gray1, gray2, None,
                                                pyr_scale=0.5, levels=3, winsize=15,
                                                iterations=3, poly_n=5, poly_sigma=1.2, flags=0)
        fx, fy = flow[..., 0], flow[..., 1]
        return fx, fy

    @staticmethod
    def _flow_to_tensor(fx, fy, size=224) -> torch.Tensor:
        mag, ang = cv2.cartToPolar(fx.astype("float32"), fy.astype("float32"))
        # Normalisation robuste
        eps = 1e-5
        fxn = (fx - fx.mean()) / (fx.std() + eps)
        fyn = (fy - fy.mean()) / (fy.std() + eps)
        magn = (mag - mag.mean()) / (mag.std() + eps)
        stack = torch.from_numpy(
            np.stack([fxn, fyn, magn], axis=0).astype("float32")
        )
        # resize/crop rapide
        stack = F.interpolate(stack.unsqueeze(0), size=(size, size), mode="bilinear", align_corners=False).squeeze(0)
        return stack

    def forward(self, frames: List["np.ndarray"], preprocess: PreprocessConfig) -> torch.Tensor:
        if len(frames) < 2:
            raise ValueError("Au moins 2 frames nécessaires pour le flow.")
        flow_tensors = []
        for i in range(len(frames) - 1):
            fx, fy = self._compute_tv_l1(frames[i], frames[i + 1])
            t = self._flow_to_tensor(fx, fy, size=preprocess.size)
            flow_tensors.append(t)
        x = torch.stack(flow_tensors, dim=0)  # (T-1, 3, H, W)
        x = x.to(next(self.parameters()).device)
        # Encoder chaque carte de flow puis agréger
        feats = self.encoder(x)
        feats = feats.flatten(1)  # (T-1, D)
        feat = feats.mean(dim=0)  # (D,)
        return feat.unsqueeze(0)

In [8]:
# ---------------------- Extracteur Combiné ----------------------

class CombinedExtractor(nn.Module):
    def __init__(self,
                 mvit: MViTBackbone,
                 resnet: ResNetBackbone,
                 flow: OpticalFlowEncoder,
                 preprocess_cfg: PreprocessConfig = PreprocessConfig(),
                 sampling_cfg: SamplingConfig = SamplingConfig()):
        super().__init__()
        self.mvit = mvit
        self.resnet = resnet
        self.flow = flow
        self.pre_cfg = preprocess_cfg
        self.samp_cfg = sampling_cfg

    def extract(self, video_path: str) -> Dict[str, torch.Tensor]:
        with SimpleVideoReader(video_path) as reader:
            n_total = len(reader)
            if n_total <= 0:
                raise RuntimeError(f"Vidéo vide ou illisible: {video_path}")
            idx = sample_indices(n_total, self.samp_cfg)
            if not idx:
                # fallback: prendre frame centrale
                idx = [n_total // 2]
            frames = reader.read_frames(idx)
        if len(frames) == 0:
            raise RuntimeError(f"Décodage vidéo vide pour: {video_path}")

        # MViT: empilement temporel
        clip = torch.stack([preprocess_frame(f, self.pre_cfg) for f in frames], dim=1)  # (3, T, H, W)
        clip = clip.unsqueeze(0).to(next(self.mvit.parameters()).device)  # (1,3,T,H,W)
        mvit_feat = self.mvit.forward_many(clip)  # (1, Dm)

        # ResNet central
        mid = frames[len(frames) // 2]
        central = preprocess_frame(mid, self.pre_cfg).unsqueeze(0).to(next(self.resnet.parameters()).device)
        resnet_feat = self.resnet(central)  # (1, Dr)

        # Optical flow embedding (sur les frames échantillonnées)
        if len(frames) >= 2:
            flow_feat = self.flow(frames, self.pre_cfg)  # (1, Df)
        else:
            # fallback: zéro si une seule frame
            device = next(self.flow.parameters()).device
            flow_feat = torch.zeros((1, self.flow.out_dim), device=device, dtype=resnet_feat.dtype)

        return {
            "temporal_mvit": mvit_feat,
            "central_resnet": resnet_feat,
            "optical_flow": flow_feat,
        }

In [9]:
# ---------------------- Fine-tuning / Distillation ----------------------

def unfreeze_until(module: nn.Module, n_blocks: int):
    """Dé-gèle séquentiellement les n premiers blocs d'un module ayant un attribut .layers ou .blocks.
    Fallback: dé-gèle les n premiers sous-modules list(module.children()).
    """
    for p in module.parameters():
        p.requires_grad = False
    children = []
    if hasattr(module, "layers"):
        children = list(module.layers)
    elif hasattr(module, "blocks"):
        children = list(module.blocks)
    else:
        children = list(module.children())
    for ch in children[:n_blocks]:
        for p in ch.parameters():
            p.requires_grad = True


def unfreeze_last_n(module: nn.Module, n_blocks: int):
    """Dé-gèle les n DERNIERS blocs d'un module (typique pour fine-tuning).
    Fonctionne si le module expose .layers / .blocks ou children().
    """
    for p in module.parameters():
        p.requires_grad = False
    if hasattr(module, "layers"):
        children = list(module.layers)
    elif hasattr(module, "blocks"):
        children = list(module.blocks)
    else:
        children = list(module.children())
    for ch in children[-n_blocks:]:
        for p in ch.parameters():
            p.requires_grad = True


def build_mlp(in_dim: int, out_dim: int, hidden: Tuple[int, ...] = (1024,)) -> nn.Module:
    layers = []
    d = in_dim
    for h in hidden:
        layers += [nn.Linear(d, h), nn.ReLU(inplace=True), nn.Dropout(0.1)]
        d = h
    layers += [nn.Linear(d, out_dim)]
    return nn.Sequential(*layers)


class SimpleFinetuneHead(nn.Module):
    """Concat features et MLP pour une tâche (e.g., retrieval ou classification).
    """
    def __init__(self, dims: Dict[str, int], out_dim: int):
        super().__init__()
        self.order = ["temporal_mvit", "central_resnet", "optical_flow"]
        in_dim = sum(dims[k] for k in self.order)
        self.proj = build_mlp(in_dim, out_dim)

    def forward(self, feats: Dict[str, torch.Tensor]) -> torch.Tensor:
        xs = [feats[k] for k in self.order if k in feats]
        x = torch.cat(xs, dim=1)
        return self.proj(x)

In [10]:
# ---------------------- Entraînement (squelette) ----------------------
import time
def get_trainable_params(extractor: CombinedExtractor, head: nn.Module):
    """Récupère les paramètres avec requires_grad=True pour l'optimizer."""
    params = []
    for m in [extractor.mvit, extractor.resnet, extractor.flow, head]:
        for p in m.parameters():
            if p.requires_grad:
                params.append(p)
    return params


def _stack_feature_dicts(dict_list: list) -> Dict[str, torch.Tensor]:
    """Empile une liste de dicts {k: (1,D)} en {k: (B,D)}"""
    out: Dict[str, torch.Tensor] = {}
    keys = dict_list[0].keys()
    for k in keys:
        out[k] = torch.cat([d[k] for d in dict_list], dim=0)
    return out


def train_one_epoch(extractor: CombinedExtractor,
                    head: nn.Module,
                    dataloader,
                    optimizer,
                    device: Optional[str] = None,
                    train_backbones: bool = False) -> float:
    """Train one epoch.
    Resolve device here (GPU if available) and ensure extractor and head are
    placed on that device. Features and labels are moved to device before
    forward/backward.
    """
    # Resolve device
    device = device if device is not None else ("cuda" if torch.cuda.is_available() else "cpu")

    # Best-effort move of modules to device
    try:
        extractor.to(device)
    except Exception:
        pass
    head.to(device)

    # Set training modes
    extractor.train(train_backbones)
    head.train()

    criterion = nn.CrossEntropyLoss()
    running_loss = 0.0
    n_samples = 0
    i = 0
    total_batches = len(dataloader)
    print(f"Starting epoch with {total_batches} batches to do. Device: {device}")
    
    t0 = time.time()
    for batch in dataloader:
        paths = batch["video_path"]
        labels = batch["label"].to(device)
        if isinstance(paths, str):
            paths = [paths]

        

        # Extract features (with or without grad depending on train_backbones)
        ctx = torch.enable_grad() if train_backbones else torch.no_grad()
        with ctx:
            feat_list = [extractor.extract(p) for p in paths]
            feats = _stack_feature_dicts(feat_list)

        # Move features to device (safe even if already on device)
        feats = {k: v.to(device) for k, v in feats.items()}

        optimizer.zero_grad(set_to_none=True)
        logits = head(feats)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        bsz = labels.shape[0]
        running_loss += loss.item() * bsz
        n_samples += bsz

        i += 1

        

        if i % 100 == 0:
            print(f"epoch {i} done, {time.time() - t0}s from start")

    return running_loss / max(n_samples, 1)


In [11]:
# ---------------------- Exemple d'initialisation ----------------------

def build_default(device: str = "cuda",
                  num_frames: int = 64,
                  sampling: str = "uniform",
                  mvit_agg: str = "mean",
                  unfreeze_last_mvit: int = 0,
                  unfreeze_last_resnet: int = 0,
                  unfreeze_last_flow: int = 0) -> Tuple[CombinedExtractor, Dict[str, int]]:
    mvit = MViTBackbone(pretrained=True, agg=mvit_agg).to(device)
    resnet = ResNetBackbone(arch="resnet50", pretrained=True).to(device)
    flow = OpticalFlowEncoder(pretrained=True).to(device)

    # Freeze all by default
    for m in [mvit, resnet, flow]:
        for p in m.parameters():
            p.requires_grad = False
    # Optionally unfreeze last N blocks for fine-tuning
    if unfreeze_last_mvit > 0:
        unfreeze_last_n(mvit.model, unfreeze_last_mvit)
    if unfreeze_last_resnet > 0:
        unfreeze_last_n(resnet.encoder, unfreeze_last_resnet)
    if unfreeze_last_flow > 0:
        unfreeze_last_n(flow.encoder, unfreeze_last_flow)

    extractor = CombinedExtractor(
        mvit=mvit,
        resnet=resnet,
        flow=flow,
        preprocess_cfg=PreprocessConfig(size=224),
        sampling_cfg=SamplingConfig(num_frames=num_frames, strategy=sampling),
    )

    # Ensure the CombinedExtractor module (and its submodules) is on the requested device
    try:
        extractor.to(device)
    except Exception:
        pass

    # Probe dims
    # Caller may do a real probe, but provide best-known dims here if available
    dims = {
        "temporal_mvit": getattr(mvit, "feature_dim", 768),
        "central_resnet": getattr(resnet, "out_dim", 2048),
        "optical_flow": getattr(flow, "out_dim", 512),
    }
    return extractor, dims


In [28]:

import json, glob, os
import time
from torch.utils.data import DataLoader, Dataset

# Try to locate a local videodatainfo JSON first
candidates = [
    os.path.join(os.getcwd(), "train_val_videodatainfo.json"),
    os.path.join(os.getcwd(), "videodatainfo.json"),
]
JSON_PATH = next((p for p in candidates if os.path.isfile(p)), None)

train_videos = []
if JSON_PATH is not None:
    with open(JSON_PATH, "r") as f:
        data = json.load(f)
    vids = data.get("videos", data if isinstance(data, list) else [])
    for v in vids:
        if v.get("split", "train") != "train":
            continue
        # Resolve a usable video path
        vp = v.get("video_path") or v.get("path") or v.get("filepath")
        if not vp:
            vid_id = v.get("video_id") or v.get("id") or v.get("name")
            if vid_id:
                vp = os.path.join(os.getcwd(), "videos", f"{vid_id}.mp4")
        if vp and not os.path.isabs(vp):
            vp = os.path.join(os.getcwd(), vp)
        if vp and os.path.isfile(vp):
            try:
                lbl = int(v.get("category", 0))
            except Exception:
                lbl = 0
            train_videos.append({"video_path": vp, "label": lbl})

# Fallback: scan local ./videos directory
if not train_videos:
    video_dir = os.path.join(os.getcwd(), "videos")
    mp4s = sorted(glob.glob(os.path.join(video_dir, "*.mp4")))
    train_videos = [{"video_path": p, "label": 0} for p in mp4s]

print(f"Found {len(train_videos)} training videos.")

# Minimal dataset
class MSRVTTDataset(Dataset):
    def __init__(self, video_list):
        self.videos = video_list
    def __len__(self):
        return len(self.videos)
    def __getitem__(self, idx):
        v = self.videos[idx]
        return {
            "video_path": v["video_path"],
            "label": torch.tensor(v["label"], dtype=torch.long),
        }

device = "cuda" if torch.cuda.is_available() else "cpu"
try:
    extractor, est_dims = build_default(device=device, num_frames=64, sampling="uniform",
                                        unfreeze_last_mvit=2, unfreeze_last_resnet=2, unfreeze_last_flow=2)
except ImportError as e:
    raise RuntimeError(
        "Missing dependency for MViT or OpenCV. Please install: pip install pytorchvideo opencv-contrib-python"
    ) from e

# Probe feature dims from a real sample to avoid mismatches
assert len(train_videos) > 0, "No videos found under JSON or ./videos"
with torch.no_grad():
    probe_feats = extractor.extract(train_videos[0]["video_path"])
dims = {k: v.shape[1] for k, v in probe_feats.items()}

# Determine number of classes from labels, default to 1 if unknown
uniq = sorted({int(x["label"]) for x in train_videos})
out_dim = (max(uniq) + 1) if uniq else 1
print(out_dim)
head = SimpleFinetuneHead(dims, out_dim=out_dim).to(device)

dataset = MSRVTTDataset(train_videos)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)#), num_workers=12)

# Build optimizer over all trainable params (backbones + head)
params = get_trainable_params(extractor, head)
optimizer = torch.optim.AdamW(params, lr=1e-4)

total_epochs = 10
epoch_times = []
def _format_time(s):
    # s in seconds -> H:MM:SS
    s = int(round(s))
    m, s = divmod(s, 60)
    h, m = divmod(m, 60)
    return f"{h}:{m:02d}:{s:02d}"
print("Starting first train")
for epoch in range(1, total_epochs + 1):
    # Train one epoch with backbones finetuning enabled
    t0 = time.time()
    loss = train_one_epoch(extractor, head, dataloader, optimizer, device, train_backbones=True)
    t_epoch = time.time() - t0
    epoch_times.append(t_epoch)

    avg = sum(epoch_times) / len(epoch_times)
    remaining = max(0, total_epochs - epoch)
    eta = avg * remaining

    print(f"Epoch {epoch}/{total_epochs} - loss: {loss:.4f} - time: {_format_time(t_epoch)} - avg: {_format_time(avg)} - ETA: {_format_time(eta)}")

    # Save tuned models and head
    save_dir = os.path.join(os.getcwd(), "checkpoints")
    os.makedirs(save_dir, exist_ok=True)
    torch.save(extractor.mvit.state_dict(), os.path.join(save_dir, "mvit_tuned.pt"))
    torch.save(extractor.resnet.state_dict(), os.path.join(save_dir, "resnet_tuned.pt"))
    torch.save(extractor.flow.state_dict(), os.path.join(save_dir, "flow_tuned.pt"))
    torch.save(head.state_dict(), os.path.join(save_dir, "head_tuned.pt"))
    print(f"Saved checkpoints to {save_dir}")

Found 6513 training videos.
20
Starting first train
Starting epoch with 815 batches to do. Device: cuda
epoch 100 done, 1042.8432984352112s from start
epoch 200 done, 2084.9936220645905s from start
epoch 300 done, 3128.588493824005s from start
epoch 400 done, 4171.013853311539s from start
epoch 500 done, 5216.94039440155s from start
epoch 600 done, 6257.541187047958s from start
epoch 700 done, 7299.544810295105s from start
epoch 800 done, 8341.062346696854s from start
Epoch 1/10 - loss: 2.0082 - time: 2:21:28 - avg: 2:21:28 - ETA: 21:13:15
Saved checkpoints to d:\Projet Donnees Multimedia\Projet-Donnees-Multimedia-SID\checkpoints
Starting epoch with 815 batches to do. Device: cuda
epoch 100 done, 1034.2052476406097s from start
epoch 200 done, 2068.551950931549s from start
epoch 300 done, 3099.5945115089417s from start
epoch 400 done, 4138.097864627838s from start
epoch 500 done, 5177.684540271759s from start
epoch 600 done, 6214.445602893829s from start
epoch 700 done, 7249.67542195320

In [12]:
# --- Nouvelle version de MViTBackbone : retourne les 4 embeddings si demandé ---

class MViTBackbone(nn.Module):
    """
    MViT backbone modifié :
    - Retourne par défaut la moyenne (comportement inchangé)
    - Si return_segments=True -> retourne les 4 embeddings (1, 4, 768)
    """
    def __init__(self, feature_dim: int = 768, pretrained: bool = True, agg: Literal["mean", "max"] = "mean"):
        super().__init__()
        if mvit_base_16x4 is None:
            raise ImportError("pytorchvideo non disponible. Installez pytorchvideo pour MViT.")
        self.model = mvit_base_16x4(pretrained=pretrained)
        if hasattr(self.model, "head") and hasattr(self.model.head, "proj"):
            self.model.head.proj = nn.Identity()
        self.agg = agg
        self.feature_dim = feature_dim

    def forward_clip16(self, clip: torch.Tensor) -> torch.Tensor:
        out = self.model(clip)
        if isinstance(out, (list, tuple)):
            out = out[0]
        return out

    def forward_many(self, clip: torch.Tensor, return_segments: bool = False) -> torch.Tensor:
        B, C, T, H, W = clip.shape
        chunks = []
        for start in range(0, T, 16):
            seg = clip[:, :, start:start + 16, :, :]
            if seg.shape[2] < 16:
                pad = 16 - seg.shape[2]
                last = seg[:, :, -1:, :, :].repeat(1, 1, pad, 1, 1)
                seg = torch.cat([seg, last], dim=2)
            feat = self.forward_clip16(seg)
            chunks.append(feat)
        feats = torch.stack(chunks, dim=1)  # (B, num_segments, D)

        if return_segments:
            return feats  # (B, 4, 768) typiquement
        if self.agg == "mean":
            return feats.mean(dim=1)
        else:
            return feats.max(dim=1).values


In [14]:
# --- Chargement des modèles fine-tunés ---

# Chemins vers les checkpoints
ckpt_mvit = "checkpoints/mvit_tuned.pt"
ckpt_resnet = "checkpoints/resnet_tuned.pt"
ckpt_flow = "checkpoints/flow_tuned.pt"
# Instanciation des modèles
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

mvit = MViTBackbone(pretrained=False).to(device)
resnet = ResNetBackbone("resnet50", pretrained=False).to(device)
flow = OpticalFlowEncoder(pretrained=False).to(device)

# Chargement des poids
mvit.load_state_dict(torch.load(ckpt_mvit, map_location=device))
resnet.load_state_dict(torch.load(ckpt_resnet, map_location=device))
flow.load_state_dict(torch.load(ckpt_flow, map_location=device))

mvit.eval()
resnet.eval()
flow.eval()

print("✅ Modèles fine-tunés chargés avec succès.")


✅ Modèles fine-tunés chargés avec succès.


In [17]:
import os
import numpy as np
from tqdm import tqdm

@torch.no_grad()
def extract_and_save_features(input_dir: str, output_dir: str):
    os.makedirs(output_dir, exist_ok=True)

    video_files = [f for f in os.listdir(input_dir) if f.lower().endswith((".mp4", ".avi", ".mov", ".mkv"))]
    print(f"📁 {len(video_files)} vidéos trouvées dans {input_dir}")

    for vid_name in tqdm(video_files, desc="Extraction en cours"):
        vid_path = os.path.join(input_dir, vid_name)

        # Lecture et sampling des frames
        reader = SimpleVideoReader(vid_path)
        indices = sample_indices(len(reader), SamplingConfig(num_frames=64))
        frames = reader.read_frames(indices)
        reader.release()

        # Prétraitement
        preprocessed = torch.stack([preprocess_frame(f, PreprocessConfig()) for f in frames]).to(device)
        preprocessed = preprocessed.unsqueeze(0).permute(0, 2, 1, 3, 4)  # (1, 3, 64, H, W)

        # --- Extraction des 3 modèles ---
        # MViT : retourne 4 embeddings (1, 4, 768)
        mvit_feats = mvit.forward_many(preprocessed, return_segments=True).squeeze(0).cpu().numpy()  # (4, 768)

        # ResNet : frame centrale
        middle_idx = len(frames) // 2
        resnet_frame = preprocess_frame(frames[middle_idx], PreprocessConfig()).unsqueeze(0).to(device)
        resnet_feats = resnet(resnet_frame).cpu().numpy()  # (1, 2048)

        # Optical Flow : flux entre frames successives
        flow_feats_list = []
        for i in range(len(frames) - 1):
            fx, fy = OpticalFlowEncoder._compute_tv_l1(frames[i], frames[i + 1])
            flow_tensor = OpticalFlowEncoder._flow_to_tensor(fx, fy)
            if isinstance(flow_tensor, np.ndarray):
                flow_tensor = torch.from_numpy(flow_tensor)
            flow_tensor = flow_tensor.unsqueeze(0).to(device)
            feat = flow.encoder(flow_tensor).flatten(1)
            flow_feats_list.append(feat.cpu().numpy())
        flow_feats = np.stack(flow_feats_list, axis=0).mean(axis=0)  # moyenne (1, 512)

        # --- Fusion et sauvegarde ---
        concat_feats = np.concatenate([
            mvit_feats.flatten(),  # 4*768 = 3072
            resnet_feats.flatten(),  # 2048
            flow_feats.flatten()  # 512
        ], axis=0)  # total 5632 dims

        out_name = os.path.splitext(vid_name)[0] + ".npy"
        out_path = os.path.join(output_dir, out_name)
        np.save(out_path, concat_feats)

    print(f"✅ Extraction terminée. Features sauvegardées dans {output_dir}")


In [18]:
input_dir = "./videos/"
output_dir = "./features_finetuned/"
extract_and_save_features(input_dir, output_dir)


📁 10000 vidéos trouvées dans ./videos/


Extraction en cours: 100%|██████████| 10000/10000 [3:57:47<00:00,  1.43s/it] 

✅ Extraction terminée. Features sauvegardées dans ./features_finetuned/
